# Image data and the convolution operation — Code-Along Template

**Type into this notebook during class.** The explanation is here; the code is not.

| | |
|---|---|
| **Follow along** | Write the code in each `# TODO` cell. The hint names the functions and arguments. |
| **If you fall behind** | Do not copy frantically. Skip to the next **CHECKPOINT**, run it, rejoin. |
| **If a cell errors** | Later cells depend on it and will fail too. Run the next CHECKPOINT rather than debugging under time pressure. |
| **After class** | The worked version is `D021_02_Convolution_Deep_Dive.ipynb`. Compare, do not copy. |

Cells that are already filled in are deliberate — imports and quick glances at the
data cost class time and teach nothing.

# Chapter 2: The Magic of Convolutions

In the previous chapter, we learned that treating an image as a flat vector destroys important spatial information. We also saw that Dense networks (MLPs) struggle with parameter explosion on large images.

In this notebook, we introduce the **Convolutional Neural Network (CNN)**. At its heart is the **Convolution** operation, which mimics how biological vision processing works.

**Objective:**
1.  Understand the mechanics of the Convolution operation (Math & Intuition).
2.  Visualize how different **Kernels** (Filters) extract features like edges and sharpness.
3.  Master concepts like **Padding** and **Stride**.
4.  Build a functional CNN using Keras `Conv2D` layers.

---

## 1. Summary of Concepts

Before diving into code, here is a quick reference table for the key terminology we will cover.

| Concept | Explanation | Visual Intuition |
| :--- | :--- | :--- |
| **Convolution** | A mathematical operation where a small matrix (Kernel) slides over the image to detect features. | A flashlight scanning a dark room to find specific objects. |
| **Kernel (Filter)** | A small learnable matrix (e.g., 3x3) that contains weights. Different weights detect different patterns (vertical edges, horizontal edges, etc.). | A stencil or a stamp. |
| **Stride** | The step size the kernel takes as it slides across the image. A stride of 2 means it jumps 2 pixels at a time. | Taking larger steps while walking. Larger stride = Smaller output. |
| **Padding** | Adding extra pixels (usually zeros) around the border of the image to control the output size. | Using a frame or border around a picture. |
| **Dilation** | Spreading the kernel elements further apart to cover a larger area without adding more parameters. | Using widely spaced dots to form a pattern. |
| **Feature Map** | The output of a convolution operation. It represents the "features" found by the kernel. | A map of where the edges (or other features) are located in the image. |

---

## 2. The Convolution Operation

Imagine a small window sliding over your image. At each position, we multiply the pixels in the window by the weights in our kernel and sum them up. This single number becomes a pixel in our output feature map.

![Convolution Animation](https://upload.wikimedia.org/wikipedia/commons/1/19/2D_Convolution_Animation.gif)
*Figure: A 3x3 Kernel sliding over a 5x5 input image.*
[Image Source](https://commons.wikimedia.org/)

### 2.1 Manual 2D Convolution (NumPy implementation)

Let's ignore Keras for a moment and implement this math manually to truly understand it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color

# Load a sample image and convert to grayscale for simplicity
image = color.rgb2gray(data.chelsea())

def manual_convolution(image, kernel):
    # Get dimensions
    image_h, image_w = image.shape
    kernel_h, kernel_w = kernel.shape
    
    # Calculate output dimensions (assuming valid padding, stride=1)
    output_h = image_h - kernel_h + 1
    output_w = image_w - kernel_w + 1
    
    # Initialize output
    output = np.zeros((output_h, output_w))
    
    # Slide the window
    for y in range(output_h):
        for x in range(output_w):
            # Extract the region of interest (ROI)
            roi = image[y:y+kernel_h, x:x+kernel_w]
            # Element-wise multiplication and sum
            output[y, x] = np.sum(roi * kernel)
            
    return output

print("Manual convolution function defined.")

In [ ]:
# ═══ CHECKPOINT — "Visualizing Kernels" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color
image = color.rgb2gray(data.chelsea())
def manual_convolution(image, kernel):
    image_h, image_w = image.shape
    kernel_h, kernel_w = kernel.shape
    output_h = image_h - kernel_h + 1
    output_w = image_w - kernel_w + 1
    output = np.zeros((output_h, output_w))
    for y in range(output_h):
        for x in range(output_w):
            roi = image[y:y + kernel_h, x:x + kernel_w]
            output[y, x] = np.sum(roi * kernel)
    return output

print("Checkpoint restored. Ready for: Visualizing Kernels")

## 3. Visualizing Kernels

Different grids of numbers (kernels) produce different effects. Let's try two classic ones:
1.  **Vertical Edge Detection** (Sobel)
2.  **Sharpening**

In [ ]:
# TODO: compute `vertical_edge_kernel`, `sharpen_kernel`, `edge_output`
# Name your result: vertical_edge_kernel, sharpen_kernel, edge_output, sharp_output

**Observation:**
- Notice how the "Vertical Edge" map lights up (becomes white) where there are vertical lines (like the cat's whiskers or the edge of the eye).
- **Deep Learning Insight:** In a CNN, we don't hard-code these numbers (-1, 0, 1). The network **learns** the best numbers to solve the problem! 

---

## 4. Padding and Strides

When we performed convolution above, the output image was slightly smaller than the input (by 2 pixels on each side for a 3x3 kernel). 

### 4.1 Padding
- **Valid Padding (Default):** Do not apply any padding. The output shrinks.
- **Same Padding:** Pad the input with zeros so that `output_size = input_size`.

![Padding Diagram](https://upload.wikimedia.org/wikipedia/commons/e/e2/Same-padding-convolution.gif)
*Figure: 'Same' padding adds a border to keep dimensions constant.*
[Image Source](https://commons.wikimedia.org/)

### 4.2 Strides
Stride is the step size. 
- Stride 1: Slide 1 pixel at a time.
- Stride 2: Slide 2 pixels (skip one). This halves the output size.

**Formula for Output Size ($O$):**
$$ O = \frac{I - K + 2P}{S} + 1 $$
Where $I$ is Input size, $K$ is Kernel size, $P$ is Padding, $S$ is Stride.

In [ ]:
from tensorflow.keras.layers import Conv2D, Input
from tensorflow.keras.models import Sequential

# Let's verify output shapes with Keras

# Case 1: Valid Padding, Stride 1
model_1 = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(filters=1, kernel_size=(3, 3), padding='valid', strides=(1, 1))
])
print(f"Valid Padding, Stride 1 Output: {model_1.output_shape}")
# Expect: (None, 26, 26, 1) -> 28 - 3 + 1 = 26

# Case 2: Same Padding, Stride 1
model_2 = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(filters=1, kernel_size=(3, 3), padding='same', strides=(1, 1))
])
print(f"Same Padding, Stride 1 Output:  {model_2.output_shape}")
# Expect: (None, 28, 28, 1)

# Case 3: Same Padding, Stride 2
model_3 = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(filters=1, kernel_size=(3, 3), padding='same', strides=(2, 2))
])
print(f"Same Padding, Stride 2 Output:  {model_3.output_shape}")
# Expect: (None, 14, 14, 1) -> 28 / 2 = 14

---

## 5. Building a Real CNN

Now we can construct the architecture described in the diagrams. We typically treat a CNN as two parts:
1.  **Feature Application (Conv Base):** Extracts patterns (lines, curves, etc.).
2.  **Classifier (Dense Head):** Uses those patterns to make a final decision.

![Detailed CNN Architecture](https://upload.wikimedia.org/wikipedia/commons/6/63/Typical_cnn.png)
*Figure: Standard CNN Architecture: Input -> Conv + ReLU -> Flatten -> Dense -> Output.*
[Image Source](https://commons.wikimedia.org/)

In [ ]:
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Flatten, Dense

# 1. Prepared Data (same as previous notebook, but we keep the 3D shape)
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# Scale to 0-1
train_images = train_images / 255.0
test_images = test_images / 255.0

# Reshape to include channel dimension (28, 28, 1)
# This is CRITICAL for Conv2D layers
train_images = train_images.reshape((-1, 28, 28, 1))
test_images = test_images.reshape((-1, 28, 28, 1))

train_labels_oh = to_categorical(train_labels)
test_labels_oh = to_categorical(test_labels)

# 2. Build the Model
model = Sequential([
    # Conv Layer 1: Detect 32 different features using 3x3 kernels
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    # Output here will be (26, 26, 32)
    
    # Flatten the 3D cube features into a 1D vector
    Flatten(),
    
    # Dense Classifier
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# TODO: compute `history`
# Name your result: history

In [ ]:
# ═══ CHECKPOINT — "Dilated Convolutions" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import numpy as np
import matplotlib.pyplot as plt
from skimage import data, color
image = color.rgb2gray(data.chelsea())
def manual_convolution(image, kernel):
    image_h, image_w = image.shape
    kernel_h, kernel_w = kernel.shape
    output_h = image_h - kernel_h + 1
    output_w = image_w - kernel_w + 1
    output = np.zeros((output_h, output_w))
    for y in range(output_h):
        for x in range(output_w):
            roi = image[y:y + kernel_h, x:x + kernel_w]
            output[y, x] = np.sum(roi * kernel)
    return output
vertical_edge_kernel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
edge_output = manual_convolution(image, vertical_edge_kernel)
sharp_output = manual_convolution(image, sharpen_kernel)
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
from tensorflow.keras.layers import Conv2D, Input
from tensorflow.keras.models import Sequential
model_1 = Sequential([Input(shape=(28, 28, 1)), Conv2D(filters=1, kernel_size=(3, 3), padding='valid', strides=(1, 1))])
model_2 = Sequential([Input(shape=(28, 28, 1)), Conv2D(filters=1, kernel_size=(3, 3), padding='same', strides=(1, 1))])
model_3 = Sequential([Input(shape=(28, 28, 1)), Conv2D(filters=1, kernel_size=(3, 3), padding='same', strides=(2, 2))])
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Flatten, Dense
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()
train_images = train_images / 255.0
test_images = test_images / 255.0
train_images = train_images.reshape((-1, 28, 28, 1))
test_images = test_images.reshape((-1, 28, 28, 1))
train_labels_oh = to_categorical(train_labels)
test_labels_oh = to_categorical(test_labels)
model = Sequential([Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)), Flatten(), Dense(128, activation='relu'), Dense(10, activation='softmax')])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train_images, train_labels_oh, epochs=1, validation_split=0.1)

print("Checkpoint restored. Ready for: Dilated Convolutions")

## 6. Dilated Convolutions

Finally, a brief mention of **Dilated Convolutions**. This is where we space out the kernel points. It allows the kernel to see a wider area (larger receptive field) without increasing the number of parameters.

![Dilated Convolution](https://raw.githubusercontent.com/vdumoulin/conv_arithmetic/master/gif/dilation.gif)
*Figure: Dilated convolution showing gaps in the kernel.*
[Image Source](https://github.com/vdumoulin/conv_arithmetic)


In [ ]:
# TODO: compute `dilated_layer`
# Name your result: dilated_layer